# Bobby Carrot — PPO Training (Train L1–25, Test L26–30)

Train step-by-step on **normal levels 1–25**, test generalization on **levels 26–30**.

Curriculum is split into five explicit phases so each mechanic group is mastered
before the next is introduced:

| Phase | Levels | New mechanics |
|-------|--------|---------------|
| **Phase 1** | 1–5   | Floor, carrot, crumble |
| **Phase 2** | 1–10  | + Conveyor belts (L/R/U/D) |
| **Phase 3** | 1–15  | + Arrows, bi-directional conveyors |
| **Phase 4** | 1–20  | + Keys, locks, switches |
| **Phase 5** | 1–25  | All mechanics, dense levels |
| **Test**    | 26–30 | Unseen harder variants of all mechanics |

Each phase resumes from the previous best checkpoint and ends with an automatic
evaluation before proceeding.

## 1. Setup — Clone & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Charan20510/Mini_Project_Game.git /content/Mini_Project_Game
%cd /content/Mini_Project_Game

In [ ]:
%pip install torch numpy pandas matplotlib --quiet

In [ ]:
import os, sys
os.chdir('/content/Mini_Project_Game')
sys.path.insert(0, '/content/Mini_Project_Game')
sys.path.insert(0, '/content/Mini_Project_Game/Game_Python')
print('Working dir:', os.getcwd())

In [ ]:
# ── Helper: copy previous phase checkpoints from Drive (skip if running fresh) ──
DRIVE_ROOT = '/content/drive/MyDrive/Bobby_Carrot_RL'

def restore_phase(phase_num: int) -> bool:
    """Copy checkpoints_phaseN from Drive if they exist. Returns True if found."""
    src = f'{DRIVE_ROOT}/checkpoints_phase{phase_num}'
    dst = f'/content/Mini_Project_Game/checkpoints_phase{phase_num}'
    if os.path.isdir(src):
        import shutil
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'✅ Restored checkpoints_phase{phase_num} from Drive')
        return True
    print(f'ℹ️  No Drive backup found for phase {phase_num} — will train from scratch')
    return False

def save_phase(phase_num: int) -> None:
    """Back up checkpoints_phaseN and logs_phaseN to Drive."""
    import shutil
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    for kind in ('checkpoints', 'logs'):
        src = f'/content/Mini_Project_Game/{kind}_phase{phase_num}'
        dst = f'{DRIVE_ROOT}/{kind}_phase{phase_num}'
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f'💾 Saved {kind}_phase{phase_num} → Drive')

def best_ckpt(phase_num: int) -> str:
    best = f'checkpoints_phase{phase_num}/ppo/ppo_best.pt'
    final = f'checkpoints_phase{phase_num}/ppo/ppo_final.pt'
    if os.path.exists(best):
        return best
    if os.path.exists(final):
        print(f'⚠️  ppo_best.pt not found — falling back to ppo_final.pt')
        return final
    raise FileNotFoundError(f'No checkpoint found for phase {phase_num}')

---
## 2. Phase 1 — Levels 1–5 (Floor + Carrot + Crumble)

Trains from scratch. Curriculum starts with levels 1–3 then gates unlocking
level 4 and 5 on ≥55% success.

In [ ]:
from pathlib import Path
from Bobby_Carrot.rl_models.config import PPOConfig, TrainingConfig, ICMConfig, LevelConfig
from Bobby_Carrot.rl_models.ppo import train_ppo

level_cfg_1 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 6)],
    test_levels =[('normal', i) for i in range(1, 6)],
)
train_cfg_1 = TrainingConfig(
    total_timesteps=500_000,
    device='auto',
    checkpoint_dir=Path('checkpoints_phase1'),
    log_dir=Path('logs_phase1'),
    curriculum=True,
    curriculum_start_levels=3,
    curriculum_promotion_window=100,
    curriculum_promotion_threshold=0.40,   # P3: lowered from 0.55
    curriculum_add_levels=1,
    level_history_window=60,
    curriculum_mastery_floor=0.55,
    curriculum_min_quota=0.15,
    curriculum_dwell_windows=2,
    curriculum_fallback_threshold=0.20,    # P4: fallback so L4/L5 still get exposure
    curriculum_fallback_windows=4,
    entropy_boost_steps=50_000,
    entropy_boost_multiplier=2.0,
    regression_trigger_drop=0.25,          # P6: re-arm entropy on regression
    lr_decay_final_fraction=0.25,
    lr_decay_min_multiplier=0.3,
    observation_mode='full',
    max_steps_per_episode=600,
    reward_scale=1.0,
    reset_policy_head_on_resume=False,
)
ppo_cfg_1 = PPOConfig(
    lr=3e-4,
    entropy_coeff=0.15,
    entropy_min=0.08,                      # P6: raised from 0.04
    clip_ratio=0.2,
    rollout_length=4096,
    minibatch_size=128,
    n_epochs=4,
    cnn_channels=[32, 64, 64, 64],
)
icm_cfg_1 = ICMConfig(enabled=True, intrinsic_reward_scale=0.01)

print('Starting Phase 1 — L1–5, 500k steps, training from scratch...')
train_ppo(ppo_cfg_1, train_cfg_1, level_cfg_1, icm_cfg_1)

In [ ]:
# Evaluate Phase 1
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt1 = best_ckpt(1)
print(f'Evaluating: {ckpt1}')
r1 = evaluate_agent(
    algo='ppo', checkpoint_path=ckpt1,
    levels=[('normal', i) for i in range(1, 6)],
    episodes_per_level=20, max_steps=600,
    observation_mode='full', device_str='auto',
)
agg1 = r1['aggregate']
print(f'\nPhase 1 aggregate: {agg1["success_rate"]:.1%} success')
phase1_ok = agg1['success_rate'] >= 0.50
print('Phase 1 ready for Phase 2?', '✅ YES' if phase1_ok else '❌ NO — consider more training')
save_phase(1)

---
## 3. Phase 2 — Levels 1–10 (+ Conveyor Belts)

Resumes from Phase 1 best checkpoint. Levels 8–10 introduce conveyor belts
(forces-direction tiles). Curriculum unlocks them progressively.

In [ ]:
restore_phase(1)  # no-op if already present
resume_1 = best_ckpt(1)

level_cfg_2 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 11)],
    test_levels =[('normal', i) for i in range(1, 11)],
)
train_cfg_2 = TrainingConfig(
    total_timesteps=800_000,
    device='auto',
    checkpoint_dir=Path('checkpoints_phase2'),
    log_dir=Path('logs_phase2'),
    curriculum=True,
    curriculum_start_levels=5,       # L1-5 already mastered; gate L6-10
    curriculum_promotion_window=100,
    curriculum_promotion_threshold=0.40,   # P3
    curriculum_add_levels=2,
    level_history_window=70,
    curriculum_mastery_floor=0.55,
    curriculum_min_quota=0.12,
    curriculum_dwell_windows=2,
    curriculum_fallback_threshold=0.20,    # P4
    curriculum_fallback_windows=4,
    entropy_boost_steps=60_000,
    entropy_boost_multiplier=2.0,
    regression_trigger_drop=0.25,          # P6
    lr_decay_final_fraction=0.25,
    lr_decay_min_multiplier=0.3,
    observation_mode='full',
    max_steps_per_episode=800,
    reward_scale=1.0,
    reset_policy_head_on_resume=False,
)
ppo_cfg_2 = PPOConfig(
    lr=1e-4,                  # lower LR — warm start from phase 1
    entropy_coeff=0.12,
    entropy_min=0.08,                      # P6
    clip_ratio=0.15,
    rollout_length=4096,
    minibatch_size=128,
    n_epochs=4,
    cnn_channels=[32, 64, 64, 64],
)
icm_cfg_2 = ICMConfig(enabled=True, intrinsic_reward_scale=0.01)

print(f'Resuming Phase 2 from: {resume_1}')
print('Phase 2 — L1-10, 800k steps, introduces conveyor belts...')
train_ppo(ppo_cfg_2, train_cfg_2, level_cfg_2, icm_cfg_2, resume_path=resume_1)

In [ ]:
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt2 = best_ckpt(2)
r2 = evaluate_agent(
    algo='ppo', checkpoint_path=ckpt2,
    levels=[('normal', i) for i in range(1, 11)],
    episodes_per_level=20, max_steps=800,
    observation_mode='full', device_str='auto',
    forgetting_levels=[('normal', i) for i in range(1, 6)],
)
agg2 = r2['aggregate']
l1_5 = [r2['per_level'].get(f'normal-{i:02d}', {}).get('success_rate', 0) for i in range(1, 6)]
retention = sum(l1_5) / len(l1_5)
print(f'\nPhase 2 aggregate: {agg2["success_rate"]:.1%} | L1-5 retention: {retention:.1%}')
phase2_ok = agg2['success_rate'] >= 0.40 and retention >= 0.50
print('Phase 2 ready for Phase 3?', '✅ YES' if phase2_ok else '❌ NO — check forgetting or retrain')
save_phase(2)

---
## 4. Phase 3 — Levels 1–15 (+ Arrows & Bi-directional Conveyors)

Levels 14–15 introduce rotating arrow tiles and paired bi-directional conveyors.

In [ ]:
restore_phase(2)
resume_2 = best_ckpt(2)

level_cfg_3 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 16)],
    test_levels =[('normal', i) for i in range(1, 16)],
)
train_cfg_3 = TrainingConfig(
    total_timesteps=800_000,
    device='auto',
    checkpoint_dir=Path('checkpoints_phase3'),
    log_dir=Path('logs_phase3'),
    curriculum=True,
    curriculum_start_levels=10,      # L1-10 warm; gate L11-15
    curriculum_promotion_window=110,
    curriculum_promotion_threshold=0.40,   # P3
    curriculum_add_levels=2,
    level_history_window=75,
    curriculum_mastery_floor=0.55,
    curriculum_min_quota=0.10,
    curriculum_dwell_windows=2,
    curriculum_fallback_threshold=0.20,    # P4
    curriculum_fallback_windows=4,
    entropy_boost_steps=70_000,
    entropy_boost_multiplier=2.0,
    regression_trigger_drop=0.25,          # P6
    lr_decay_final_fraction=0.25,
    lr_decay_min_multiplier=0.3,
    observation_mode='full',
    max_steps_per_episode=1000,
    reward_scale=1.0,
    reset_policy_head_on_resume=False,
)
ppo_cfg_3 = PPOConfig(
    lr=8e-5,
    entropy_coeff=0.10,
    entropy_min=0.08,                      # P6
    clip_ratio=0.15,
    rollout_length=4096,
    minibatch_size=128,
    n_epochs=4,
    cnn_channels=[32, 64, 64, 64],
)
icm_cfg_3 = ICMConfig(enabled=True, intrinsic_reward_scale=0.01)

print(f'Resuming Phase 3 from: {resume_2}')
print('Phase 3 — L1-15, 800k steps, introduces arrows + bi-dir conveyors...')
train_ppo(ppo_cfg_3, train_cfg_3, level_cfg_3, icm_cfg_3, resume_path=resume_2)

In [ ]:
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt3 = best_ckpt(3)
r3 = evaluate_agent(
    algo='ppo', checkpoint_path=ckpt3,
    levels=[('normal', i) for i in range(1, 16)],
    episodes_per_level=20, max_steps=1000,
    observation_mode='full', device_str='auto',
    forgetting_levels=[('normal', i) for i in range(1, 6)],
)
agg3 = r3['aggregate']
print(f'\nPhase 3 aggregate: {agg3["success_rate"]:.1%}')
phase3_ok = agg3['success_rate'] >= 0.35
print('Phase 3 ready for Phase 4?', '✅ YES' if phase3_ok else '❌ NO — consider more training')
save_phase(3)

---
## 5. Phase 4 — Levels 1–20 (+ Keys, Locks, Switches)

Levels 16–20 introduce red/green switches and key/lock gates.

In [ ]:
restore_phase(3)
resume_3 = best_ckpt(3)

level_cfg_4 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 21)],
    test_levels =[('normal', i) for i in range(1, 21)],
)
train_cfg_4 = TrainingConfig(
    total_timesteps=800_000,
    device='auto',
    checkpoint_dir=Path('checkpoints_phase4'),
    log_dir=Path('logs_phase4'),
    curriculum=True,
    curriculum_start_levels=15,
    curriculum_promotion_window=115,
    curriculum_promotion_threshold=0.40,   # P3
    curriculum_add_levels=2,
    level_history_window=80,
    curriculum_mastery_floor=0.55,
    curriculum_min_quota=0.10,
    curriculum_dwell_windows=2,
    curriculum_fallback_threshold=0.20,    # P4
    curriculum_fallback_windows=4,
    entropy_boost_steps=75_000,
    entropy_boost_multiplier=2.0,
    regression_trigger_drop=0.25,          # P6
    lr_decay_final_fraction=0.25,
    lr_decay_min_multiplier=0.3,
    observation_mode='full',
    max_steps_per_episode=1200,
    reward_scale=1.0,
    reset_policy_head_on_resume=False,
)
ppo_cfg_4 = PPOConfig(
    lr=6e-5,
    entropy_coeff=0.08,
    entropy_min=0.08,                      # P6 — floor = current coeff, sustains exploration
    clip_ratio=0.12,
    rollout_length=4096,
    minibatch_size=128,
    n_epochs=4,
    cnn_channels=[32, 64, 64, 64],
)
icm_cfg_4 = ICMConfig(enabled=True, intrinsic_reward_scale=0.01)

print(f'Resuming Phase 4 from: {resume_3}')
print('Phase 4 — L1-20, 800k steps, introduces keys + locks + switches...')
train_ppo(ppo_cfg_4, train_cfg_4, level_cfg_4, icm_cfg_4, resume_path=resume_3)

In [ ]:
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt4 = best_ckpt(4)
r4 = evaluate_agent(
    algo='ppo', checkpoint_path=ckpt4,
    levels=[('normal', i) for i in range(1, 21)],
    episodes_per_level=20, max_steps=1200,
    observation_mode='full', device_str='auto',
    forgetting_levels=[('normal', i) for i in range(1, 6)],
)
agg4 = r4['aggregate']
print(f'\nPhase 4 aggregate: {agg4["success_rate"]:.1%}')
phase4_ok = agg4['success_rate'] >= 0.30
print('Phase 4 ready for Phase 5?', '✅ YES' if phase4_ok else '❌ NO — consider more training')
save_phase(4)

---
## 6. Phase 5 — Levels 1–25 (Full Training Set)

Final training phase. Levels 21–25 combine all mechanics at high density.
Low LR to fine-tune without disturbing Phase 4 weights.

In [ ]:
restore_phase(4)
resume_4 = best_ckpt(4)

level_cfg_5 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 26)],
    test_levels =[('normal', i) for i in range(26, 31)],
)
train_cfg_5 = TrainingConfig(
    total_timesteps=600_000,
    device='auto',
    checkpoint_dir=Path('checkpoints_phase5'),
    log_dir=Path('logs_phase5'),
    curriculum=True,
    curriculum_start_levels=20,
    curriculum_promotion_window=120,
    curriculum_promotion_threshold=0.40,   # P3
    curriculum_add_levels=3,
    level_history_window=80,
    curriculum_mastery_floor=0.55,
    curriculum_min_quota=0.10,
    curriculum_dwell_windows=2,
    curriculum_fallback_threshold=0.20,    # P4
    curriculum_fallback_windows=4,
    entropy_boost_steps=80_000,
    entropy_boost_multiplier=2.0,
    regression_trigger_drop=0.25,          # P6
    lr_decay_final_fraction=0.30,
    lr_decay_min_multiplier=0.3,
    observation_mode='full',
    max_steps_per_episode=1500,
    reward_scale=1.0,
    reset_policy_head_on_resume=False,
)
ppo_cfg_5 = PPOConfig(
    lr=5e-5,              # Fine-tune LR
    entropy_coeff=0.06,
    entropy_min=0.08,                      # P6 — floor > start so entropy stays constant late
    clip_ratio=0.10,      # Tight clip — preserve Phase 4 policy
    rollout_length=4096,
    minibatch_size=128,
    n_epochs=4,
    cnn_channels=[32, 64, 64, 64],
)
icm_cfg_5 = ICMConfig(enabled=True, intrinsic_reward_scale=0.01)

print(f'Resuming Phase 5 from: {resume_4}')
print('Phase 5 — L1-25, 600k steps, full mechanic set...')
train_ppo(ppo_cfg_5, train_cfg_5, level_cfg_5, icm_cfg_5, resume_path=resume_4)

In [ ]:
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt5 = best_ckpt(5)
r5_train = evaluate_agent(
    algo='ppo', checkpoint_path=ckpt5,
    levels=[('normal', i) for i in range(1, 26)],
    episodes_per_level=20, max_steps=1500,
    observation_mode='full', device_str='auto',
    forgetting_levels=[('normal', i) for i in range(1, 6)],
)
agg5 = r5_train['aggregate']
print(f'\nPhase 5 TRAIN aggregate: {agg5["success_rate"]:.1%}')
save_phase(5)

---
## 7. Final Evaluation — Test Levels 26–30 (Unseen)

The ultimate test: can the agent generalize to levels it has **never seen**?

Run both the raw PPO policy and PPO + MCTS (128 simulations) to compare.

In [ ]:
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt5 = best_ckpt(5)
TEST_LEVELS = [('normal', i) for i in range(26, 31)]

print('=' * 70)
print('  FINAL TEST (raw policy): Unseen Levels 26–30')
print('=' * 70)
results_raw = evaluate_agent(
    algo='ppo', checkpoint_path=ckpt5,
    levels=TEST_LEVELS,
    episodes_per_level=50, max_steps=1500,
    observation_mode='full', device_str='auto',
    use_mcts=False,
)

print('\n' + '=' * 70)
print('  FINAL TEST (PPO + MCTS 128): Unseen Levels 26–30')
print('=' * 70)
results_mcts = evaluate_agent(
    algo='ppo', checkpoint_path=ckpt5,
    levels=TEST_LEVELS,
    episodes_per_level=50, max_steps=1500,
    observation_mode='full', device_str='auto',
    use_mcts=True, mcts_sims=128, mcts_depth=25,
)

raw_agg  = results_raw['aggregate']
mcts_agg = results_mcts['aggregate']
target   = 0.40

print('\n' + '=' * 70)
print(f'  RAW POLICY  success: {raw_agg["success_rate"]:.1%}')
print(f'  PPO + MCTS  success: {mcts_agg["success_rate"]:.1%}')
print(f'  TARGET (≥{target:.0%}):   {"MET ✅" if mcts_agg["success_rate"] >= target else "NOT MET ❌"}')
print('=' * 70)

---
## 8. Save Final Weights to Drive

In [ ]:
import shutil

for phase_num in range(1, 6):
    save_phase(phase_num)

print(f'\nAll phases saved to {DRIVE_ROOT}')

try:
    from google.colab import drive as _drive
    _drive.flush_and_unmount()
    print('Drive flushed.')
except Exception:
    pass